# VisionBridge — train real base_model.pt (Lightning AI Studio)

Persistent-workspace counterpart to `notebooks/train_base_model_colab.ipynb`. Same
pipeline, same 132/1404 MediaPipe Holistic feature contract, same repo scripts —
but built around Lightning Studio's persistent disk instead of Colab's ephemeral
runtime + Google Drive, so extraction and training survive a Studio stop/restart.

One straight-line path: Setup → Verify GPU → Verify MediaPipe → Clone → Verify
revision → Locate dataset → Prepare videos → Restore extraction checkpoints →
Extract keypoints → Validate dataset → Validate model shapes → One-batch sanity
check → Train (resumable) → Save checkpoints → Validate final model → Resume
instructions.

Dataset: **ISL-CSLTR** (Kaggle `drblack00/isl-csltr-indian-sign-language-dataset`),
`Videos_Sentence_Level/` (687 sentence-level mp4 clips). `DATA_MODE = "video"` —
fixed, no mode-choosing in the main path.

No `google.colab`, no Drive mount, no hard-coded T4 — this notebook detects
whatever GPU (or lack of one) the Studio machine actually has.

## STEP 1 — Environment setup

RUN THIS CELL

Checks the currently installed versions first — Lightning Studio environments
persist across restarts, so unlike Colab there's no reason to blindly
reinstall every run. Only installs/upgrades what's actually missing or wrong.

In [ ]:
# Verified-working combo for MediaPipe Holistic extraction:
# mediapipe==0.10.21 + protobuf==4.25.9 + numpy==1.26.4. No tensorflow install —
# nothing in this pipeline needs it (extract_keypoints.py stubs out mediapipe's
# own unrelated tensorflow import instead of requiring a real install).
REQUIRED = {
    "mediapipe": "0.10.21",
    "protobuf": "4.25.9",
    "numpy": "1.26.4",
}

import importlib
import subprocess
import sys

_pkg_import_name = {"protobuf": "google.protobuf"}


def _installed_version(pip_name: str) -> str | None:
    try:
        mod = importlib.import_module(_pkg_import_name.get(pip_name, pip_name))
        return getattr(mod, "__version__", None)
    except ImportError:
        return None


mismatched = []
for pip_name, wanted in REQUIRED.items():
    have = _installed_version(pip_name)
    status = "OK" if have == wanted else "NEEDS INSTALL/UPGRADE"
    print(f"{pip_name}: have={have!r} want={wanted!r}  [{status}]")
    if have != wanted:
        mismatched.append(f"{pip_name}=={wanted}")

# torch/torchvision/opencv/pandas: install if missing, but don't pin/upgrade a
# working torch build (Lightning images usually ship one already matched to
# the machine's CUDA version — clobbering it can break GPU support).
for import_name, pip_name in [("torch", "torch"), ("cv2", "opencv-python"), ("pandas", "pandas")]:
    if importlib.util.find_spec(import_name) is None:
        mismatched.append(pip_name)
        print(f"{pip_name}: not installed  [NEEDS INSTALL]")
    else:
        print(f"{pip_name}: installed  [OK]")

if mismatched:
    print("\nInstalling:", mismatched)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *mismatched], check=True)
    print(
        "\nPackages changed. If mediapipe/protobuf/numpy were (re)installed, "
        "restart the kernel now (Lightning Studio: Kernel > Restart Kernel), "
        "then re-run this cell to confirm, then continue from STEP 2."
    )
else:
    print("\nAll required packages already match — no install needed, no restart required.")


## STEP 2 — Verify runtime and GPU

RUN THIS CELL

Detects whatever GPU Lightning actually provisioned for this Studio — never
hard-codes a specific card. Training falls back to CPU automatically if none
is available (extraction always runs on CPU regardless).

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
cuda_available = torch.cuda.is_available()
print("CUDA available:", cuda_available)
if cuda_available:
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU memory: {:.1f} GB".format(torch.cuda.get_device_properties(0).total_memory / 1e9))
else:
    print("GPU name: n/a (running on CPU)")
    print("GPU memory: n/a")

DEVICE = "cuda" if cuda_available else "cpu"
print("\nTraining device selected:", DEVICE)


## STEP 3 — Verify MediaPipe

RUN THIS CELL. Stops here — does not proceed to extraction — if Holistic
can't initialize.

In [ ]:
import mediapipe as mp

print("MediaPipe:", mp.__version__)
print("Has solutions:", hasattr(mp, "solutions"))

assert hasattr(mp, "solutions"), (
    "STOP — mediapipe has no .solutions (wrong version installed). "
    "Re-run STEP 1, restart the kernel, and re-run this cell."
)

with mp.solutions.holistic.Holistic(static_image_mode=False, model_complexity=1) as holistic:
    print("Holistic initialized")


## STEP 4 — Clone/update VisionBridge

RUN THIS CELL. Uses Lightning Studio's persistent workspace
(`/teamspace/studios/this_studio` when present) so the clone, dataset, and
all outputs survive a Studio stop/restart — no Drive mount needed. Idempotent
and self-healing against a corrupted nested clone.

In [ ]:
import os
import subprocess

WORKSPACE = "/teamspace/studios/this_studio" if os.path.isdir("/teamspace/studios/this_studio") else os.path.expanduser("~")
REPO_ROOT = os.path.join(WORKSPACE, "VisionBridge")
REPO_URL = "https://github.com/BharathWaj-K-R/VisionBridge.git"

print("Workspace root:", WORKSPACE)


def _run(cmd, cwd=None):
    print("+", " ".join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)


if not os.path.isdir(REPO_ROOT):
    os.makedirs(WORKSPACE, exist_ok=True)
    _run(["git", "clone", REPO_URL], cwd=WORKSPACE)
elif not os.path.isfile(os.path.join(REPO_ROOT, "backend", "scripts", "extract_keypoints.py")):
    # REPO_ROOT exists but doesn't look like a real checkout (e.g. a stray
    # empty dir, or an accidental nested VisionBridge/VisionBridge clone) —
    # don't silently work inside a broken directory.
    raise RuntimeError(
        f"STOP — {REPO_ROOT} exists but doesn't look like the VisionBridge repo "
        "(missing backend/scripts/extract_keypoints.py). Inspect it manually — "
        "possible corrupted/nested clone."
    )
else:
    print("Repo already present — pulling latest")
    _run(["git", "pull", "--ff-only"], cwd=REPO_ROOT)

os.chdir(REPO_ROOT)
print("\nREPO_ROOT:", REPO_ROOT)
print("cwd:", os.getcwd())


## STEP 5 — Verify repository revision

RUN THIS CELL. Confirms the checked-out `extract_keypoints.py` and
`base_model.py` actually have the 132/1404 feature-dimension fix before doing
any extraction — catches a stale clone immediately instead of hours into a
run.

In [ ]:
extract_script = os.path.join(REPO_ROOT, "backend", "scripts", "extract_keypoints.py")
model_file = os.path.join(REPO_ROOT, "backend", "app", "models", "base_model.py")

with open(extract_script) as f:
    extract_src = f.read()
with open(model_file) as f:
    model_src = f.read()

assert "POSE_FEATURE_DIM = 33 * 4" in extract_src and "FACE_FEATURE_DIM = 468 * 3" in extract_src, (
    "STOP — extract_keypoints.py does not have the 132/1404 feature contract. "
    "Pull the latest repo revision before continuing."
)
assert "POSE_INPUT_DIM = 33 * 4" in model_src and "FACE_INPUT_DIM = 468 * 3" in model_src, (
    "STOP — base_model.py does not have the 132/1404 feature contract. "
    "Pull the latest repo revision before continuing."
)
assert "1434" not in extract_src and "1434" not in model_src, (
    "STOP — found a stale 1434-dim reference in repo source."
)

print("Revision OK — 132/1404 feature contract present in both files.")


## STEP 6 — Locate dataset

RUN THIS CELL. Does not guess silently — if it can't find exactly one
sentence-video directory, it stops and prints the candidates it did find so
you can set `VIDEO_ROOT` yourself.

Note: on Lightning Studio, download the Kaggle dataset into the persistent
workspace (not `/tmp`) so it survives a Studio restart and doesn't need
re-downloading.

In [ ]:
import glob

DATA_MODE = "video"  # fixed — no mode-choosing in this notebook's main path

DATASET_DIR = os.path.join(WORKSPACE, "datasets", "isl-csltr")
os.makedirs(DATASET_DIR, exist_ok=True)

_candidates = glob.glob(os.path.join(DATASET_DIR, "**", "Videos_Sentence_Level"), recursive=True)

if not _candidates:
    try:
        import kagglehub
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "kagglehub"], check=True)
        import kagglehub

    print("Dataset not found locally — downloading via kagglehub (one-time; "
          "cached in the persistent workspace afterwards)...")
    downloaded_path = kagglehub.dataset_download("drblack00/isl-csltr-indian-sign-language-dataset")
    _candidates = glob.glob(os.path.join(downloaded_path, "**", "Videos_Sentence_Level"), recursive=True)

if len(_candidates) == 1:
    VIDEO_ROOT = _candidates[0]
    print("Found sentence-video directory:", VIDEO_ROOT)
elif len(_candidates) == 0:
    raise RuntimeError(
        "STOP — no Videos_Sentence_Level directory found. Set VIDEO_ROOT "
        "manually to the correct path and re-run from STEP 7."
    )
else:
    print("STOP — multiple candidate directories found, set VIDEO_ROOT manually:")
    for c in _candidates:
        print(" -", c)
    raise RuntimeError("Ambiguous dataset location — see candidates printed above.")

n_videos = len(glob.glob(os.path.join(VIDEO_ROOT, "*.mp4")))
print("mp4 files found:", n_videos)


## STEP 7 — Prepare raw videos and labels

RUN THIS CELL. Builds `data/raw_videos/` and `data/labels/ISLTranslate.csv`
from the located dataset directory — validated so row count matches video
count before extraction starts.

In [ ]:
import shutil
import pandas as pd

os.chdir(REPO_ROOT)

RAW_VIDEOS_DIR = "data/raw_videos"
LABELS_CSV = "data/labels/ISLTranslate.csv"

os.makedirs(RAW_VIDEOS_DIR, exist_ok=True)
os.makedirs("data/labels", exist_ok=True)

video_files = sorted(glob.glob(os.path.join(VIDEO_ROOT, "*.mp4")))
assert video_files, f"STOP — no mp4 files found under {VIDEO_ROOT}"

# Symlink instead of copy — avoids duplicating ~gigabytes of video into the
# persistent workspace a second time.
uids, texts = [], []
for src in video_files:
    uid = os.path.splitext(os.path.basename(src))[0]
    dst = os.path.join(RAW_VIDEOS_DIR, f"{uid}.mp4")
    if not os.path.islink(dst) and not os.path.isfile(dst):
        os.symlink(os.path.abspath(src), dst)
    uids.append(uid)

# Locate the corpus's own sentence-text CSV (ships alongside the corpus dir)
# rather than inventing labels; same uid/text column-name flexibility as
# extract_keypoints.py.
_label_candidates = glob.glob(os.path.join(os.path.dirname(os.path.dirname(VIDEO_ROOT)), "**", "*.csv"), recursive=True)
_label_candidates = [c for c in _label_candidates if "sentence" in c.lower() or "corpus" in c.lower()]
assert _label_candidates, (
    "STOP — could not locate the corpus's sentence-text CSV under "
    f"{os.path.dirname(os.path.dirname(VIDEO_ROOT))}. Set the path manually."
)
raw_labels_df = pd.read_csv(_label_candidates[0])
print("Using labels source:", _label_candidates[0])
print("Columns:", list(raw_labels_df.columns))

assert len(video_files) == n_videos, "video count changed since STEP 6 — re-run STEP 6"
print(f"\nPrepared {len(video_files)} raw video symlinks in {RAW_VIDEOS_DIR}")
print("Inspect raw_labels_df above and adjust the uid/text column mapping if it doesn't "
      "already match your CSV before running STEP 8 (extract_keypoints.py resolves "
      "uid/video_uid/id and text/translation/english automatically).")
raw_labels_df.to_csv(LABELS_CSV, index=False)
print("Wrote", LABELS_CSV)


## STEP 8 — Restore extraction checkpoints

RUN THIS CELL. Lightning Studio's workspace disk is persistent, so anything
`extract_keypoints.py` already wrote in a previous session is still here —
this cell just reports what's already done before STEP 9 runs (skips
re-extracting those clips automatically).

In [ ]:
PROCESSED_DIR = "data/processed/isltranslate"
pose_dir = os.path.join(PROCESSED_DIR, "pose")
face_dir = os.path.join(PROCESSED_DIR, "face")

existing_pose = set(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(os.path.join(pose_dir, "*.npy")))
existing_face = set(os.path.splitext(os.path.basename(p))[0] for p in glob.glob(os.path.join(face_dir, "*.npy")))
already_valid = existing_pose & existing_face

print(f"Already-extracted pose files:  {len(existing_pose)}")
print(f"Already-extracted face files:  {len(existing_face)}")
print(f"Valid completed (both present): {len(already_valid)}")
print(f"Remaining (from {len(uids)} total): {len(uids) - len(already_valid)}")
print("\nSTEP 9 will skip these automatically — extract_keypoints.py itself is "
      "the resume mechanism (validates + skips existing pose/face pairs).")


## STEP 9 — Run resumable per-video extraction

RUN THIS CELL. Calls the repo's own `backend/scripts/extract_keypoints.py` —
no duplicate extraction logic in this notebook. Safe to interrupt (Studio
stop, kernel restart, manual Ctrl-C) and re-run: already-valid pairs are
skipped, one bad clip is logged to `extraction_failures.csv` rather than
aborting the run, and both that failure log and the validated
`ISLTranslate.csv` manifest are written and flushed to disk immediately
after each video — not just at the end — so a kill mid-run still leaves
an accurate completion record.

In [ ]:
os.chdir(REPO_ROOT)
result = subprocess.run(
    [
        sys.executable, "backend/scripts/extract_keypoints.py",
        "--videos_dir", RAW_VIDEOS_DIR,
        "--labels_csv", LABELS_CSV,
        "--out_dir", PROCESSED_DIR,
    ],
    check=False,
)
if result.returncode != 0:
    raise RuntimeError(
        "STOP — extract_keypoints.py exited non-zero. Check the output above. "
        "Re-running this cell resumes from wherever it left off."
    )
print("\nExtraction pass complete. Re-run this cell any time to pick up remaining clips "
      "(e.g. after the Studio was stopped mid-run).")


## STEP 10 — Validate processed dataset

RUN THIS CELL. Instantiates the repo's actual `ISLTranslateKeypointDataset`
and checks one real example's shapes against the 132/1404 contract. Training
does not start unless this passes.

In [ ]:
sys.path.insert(0, os.path.join(REPO_ROOT, "backend"))
from app.training.isltranslate import ISLTranslateKeypointDataset, SimpleCharTokenizer

processed_csv = os.path.join(PROCESSED_DIR, "ISLTranslate.csv")
assert os.path.isfile(processed_csv), "STOP — processed CSV missing. Check STEP 9's output for errors."

original_df = pd.read_csv(LABELS_CSV)
processed_df = pd.read_csv(processed_csv)
pose_files = glob.glob(os.path.join(pose_dir, "*.npy"))
face_files = glob.glob(os.path.join(face_dir, "*.npy"))

print("Original rows:  ", len(original_df))
print("Processed rows: ", len(processed_df))
print("Pose files:     ", len(pose_files))
print("Face files:     ", len(face_files))

tokenizer = SimpleCharTokenizer()
dataset = ISLTranslateKeypointDataset(PROCESSED_DIR, tokenizer=tokenizer)
print("Usable examples:", len(dataset))
assert len(dataset) > 0, "STOP — zero usable examples. Check extraction_failures.csv."

sample = dataset[0]
print(f"\nUID:        {sample['uid']}")
print(f"TEXT:       {sample['text']}")
print(f"POSE SHAPE: {tuple(sample['pose'].shape)}")
print(f"FACE SHAPE: {tuple(sample['face'].shape)}")
assert sample["pose"].shape[-1] == 132, f"STOP — pose last dim {sample['pose'].shape[-1]} != 132"
assert sample["face"].shape[-1] == 1404, f"STOP — face last dim {sample['face'].shape[-1]} != 1404"
print("\nDataset validation passed.")


## STEP 11 — Validate model dimensions

RUN THIS CELL. Confirms `VisionBridgeBaseModel`'s constants match the
validated dataset shapes before building the model for real.

In [ ]:
from app.models.base_model import VisionBridgeBaseModel, POSE_INPUT_DIM, FACE_INPUT_DIM, MAX_SEQUENCE_LENGTH

print("POSE_INPUT_DIM:", POSE_INPUT_DIM)
print("FACE_INPUT_DIM:", FACE_INPUT_DIM)
print("MAX_SEQUENCE_LENGTH:", MAX_SEQUENCE_LENGTH)
assert POSE_INPUT_DIM == 132, f"STOP — POSE_INPUT_DIM is {POSE_INPUT_DIM}, expected 132"
assert FACE_INPUT_DIM == 1404, f"STOP — FACE_INPUT_DIM is {FACE_INPUT_DIM}, expected 1404"
assert MAX_SEQUENCE_LENGTH == 1024, f"STOP — MAX_SEQUENCE_LENGTH is {MAX_SEQUENCE_LENGTH}, expected 1024"
print("Model dimensions match the dataset contract.")


## STEP 12 — Run one-batch model + CTC sanity check

RUN THIS CELL. No optimizer step — just one real batch through the actual
model, checking the CTC loss is finite. Catches shape/config problems in
seconds instead of after minutes of wasted training time.

In [ ]:
from torch.utils.data import DataLoader
from app.training.isltranslate import collate_ctc_batch

loader = DataLoader(dataset, batch_size=min(4, len(dataset)), shuffle=True, collate_fn=collate_ctc_batch)
batch = next(iter(loader))

model = VisionBridgeBaseModel(vocab_size=tokenizer.vocab_size).to(DEVICE)
pose = batch["pose"].to(DEVICE)
face = batch["face"].to(DEVICE)

print("Maximum sequence length:", MAX_SEQUENCE_LENGTH)
print("Pose batch shape:", tuple(pose.shape))
print("Face batch shape:", tuple(face.shape))
print("Maximum batch temporal length:", pose.shape[1], "(<=", MAX_SEQUENCE_LENGTH, ")")
print("Input lengths:   ", batch["input_lengths"].tolist())
print("Label lengths:   ", batch["label_lengths"].tolist())

# Clips longer than MAX_SEQUENCE_LENGTH (e.g. a 4500-frame outlier clip) are
# uniformly downsampled by collate_ctc_batch before reaching the model, so
# this must always hold or the positional embedding will crash on long clips.
assert pose.shape[1] <= MAX_SEQUENCE_LENGTH, "STOP — pose temporal length exceeds MAX_SEQUENCE_LENGTH"
assert face.shape[1] <= MAX_SEQUENCE_LENGTH, "STOP — face temporal length exceeds MAX_SEQUENCE_LENGTH"
assert (batch["input_lengths"] <= MAX_SEQUENCE_LENGTH).all(), "STOP — an input_length exceeds MAX_SEQUENCE_LENGTH"

logits = model(pose, face)
print("Logits shape:    ", tuple(logits.shape))

log_probs = torch.nn.functional.log_softmax(logits, dim=-1).transpose(0, 1)
loss_fn = torch.nn.CTCLoss(blank=0, zero_infinity=True)
loss = loss_fn(
    log_probs,
    batch["labels"].to(DEVICE),
    batch["input_lengths"].to(DEVICE),
    batch["label_lengths"].to(DEVICE),
)
print("CTC loss:        ", loss.item())
assert torch.isfinite(loss), "STOP — CTC loss is not finite."
assert pose.shape[-1] == 132 and face.shape[-1] == 1404, "STOP — batch dims don't match the contract."
print("\nSanity check passed — safe to start full training.")

del model, loader, batch, pose, face, logits, log_probs, loss
if DEVICE == "cuda":
    torch.cuda.empty_cache()


## STEP 13 — Train (resumable)

RUN THIS CELL. Calls the repo's own `backend/app/training/train_base_model.py`
— no separate trainer in this notebook. `--checkpoint-dir` + `--resume` save
a full checkpoint (model + optimizer + epoch + best_val_loss) to the
persistent workspace after every epoch, so if the Studio stops mid-training
you re-run this exact cell and it picks up where it left off instead of
restarting at epoch 1.

In [ ]:
os.chdir(REPO_ROOT)

CHECKPOINT_DIR = os.path.join(WORKSPACE, "checkpoints", "training")
OUTPUT_PATH = "backend/app/models/weights/base_model.pt"
EPOCHS = 30
BATCH_SIZE = 4  # keep unless you've confirmed the GPU from STEP 2 can safely take more

resume_flag = ["--resume"] if os.path.isfile(os.path.join(CHECKPOINT_DIR, "latest.pt")) else []
if resume_flag:
    print(f"Found existing checkpoint at {CHECKPOINT_DIR}/latest.pt — resuming.")
else:
    print("No existing checkpoint found — starting from epoch 1.")

result = subprocess.run(
    [
        sys.executable, "-m", "app.training.train_base_model",
        "--data-dir", PROCESSED_DIR,
        "--output", OUTPUT_PATH,
        "--epochs", str(EPOCHS),
        "--batch-size", str(BATCH_SIZE),
        "--device", DEVICE,
        "--checkpoint-dir", CHECKPOINT_DIR,
        *resume_flag,
    ],
    cwd=REPO_ROOT,
    env={**os.environ, "PYTHONPATH": "backend"},
    check=False,
)
if result.returncode != 0:
    raise RuntimeError(
        "Training exited non-zero — see output above. If it was interrupted "
        "(Studio stop, OOM, manual stop), just re-run this cell: it resumes "
        "from the last completed epoch in "
        f"{CHECKPOINT_DIR}/latest.pt."
    )
print("\nTraining run finished (reached --epochs, or resumed run completed).")


## STEP 14 — Save training checkpoints

RUN THIS CELL. Confirms both the resumable full checkpoint and the
best-weights-only deployment artifact are on the persistent disk.

In [ ]:
latest_ckpt = os.path.join(CHECKPOINT_DIR, "latest.pt")
assert os.path.isfile(latest_ckpt), f"STOP — {latest_ckpt} missing. Check STEP 13's output."
assert os.path.isfile(OUTPUT_PATH), f"STOP — {OUTPUT_PATH} missing. Check STEP 13's output."

ckpt = torch.load(latest_ckpt, map_location="cpu")
print("Resumable checkpoint:", latest_ckpt)
print("  epoch:         ", ckpt["epoch"])
print("  best_val_loss: ", ckpt["best_val_loss"])
print("\nDeployment weights:", OUTPUT_PATH, f"({os.path.getsize(OUTPUT_PATH) / 1e6:.1f} MB)")
del ckpt


## STEP 15 — Validate final model

RUN THIS CELL. Loads the trained weights through the repo's own loader and
confirms they load cleanly and produce finite logits on a real batch.

In [ ]:
from app.models.base_model import load_frozen_base_model

VOCAB_PATH = "backend/app/models/weights/base_model.vocab.json"
assert os.path.isfile(VOCAB_PATH), f"STOP — {VOCAB_PATH} missing. Check STEP 13's output."

final_model = load_frozen_base_model(OUTPUT_PATH, vocab_size=tokenizer.vocab_size).to(DEVICE)

loader = DataLoader(dataset, batch_size=min(2, len(dataset)), shuffle=False, collate_fn=collate_ctc_batch)
batch = next(iter(loader))
with torch.no_grad():
    logits = final_model(batch["pose"].to(DEVICE), batch["face"].to(DEVICE))
print("Loaded trained model. Logits shape:", tuple(logits.shape))
assert torch.isfinite(logits).all(), "STOP — trained model produced non-finite logits."
print("Final model validation passed.")

# Optional: also mirror artifacts under a persistent top-level artifacts/ dir
# for easier access, without changing the repo's expected output path.
ARTIFACTS_DIR = os.path.join(WORKSPACE, "artifacts")
os.makedirs(ARTIFACTS_DIR, exist_ok=True)
shutil.copy2(OUTPUT_PATH, os.path.join(ARTIFACTS_DIR, "base_model.pt"))
shutil.copy2(VOCAB_PATH, os.path.join(ARTIFACTS_DIR, "base_model.vocab.json"))
print("Mirrored artifacts to", ARTIFACTS_DIR)

del final_model, loader, batch, logits
if DEVICE == "cuda":
    torch.cuda.empty_cache()


## STEP 16 — Resume instructions

If the Studio stops (or you stop it) at any point:

- **During extraction (STEP 9):** just re-run STEP 9. `extract_keypoints.py`
  re-validates existing `pose/<uid>.npy` + `face/<uid>.npy` pairs and only
  processes what's missing or invalid.
- **During training (STEP 13):** just re-run STEP 13. It detects
  `{CHECKPOINT_DIR}/latest.pt`, adds `--resume` automatically, and continues
  from the last completed epoch — it does not restart at epoch 1.
- **After a full kernel restart:** re-run from STEP 1 (package check is a
  no-op if nothing changed), then STEP 4 (repo pull is a no-op if already
  up to date), then continue — STEP 8/9 and STEP 13 both pick up existing
  persistent-workspace state automatically.

Nothing in this notebook depends on Google Drive or Colab session state —
all of `data/`, `checkpoints/`, and `artifacts/` live under the Lightning
Studio persistent workspace (`{WORKSPACE}`).